RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Company_Handbook.pdf
  ✓ Loaded 4 pages

Total documents loaded: 4


In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=50,chunk_overlap=10):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 43 chunks

Example chunk:
Content: Page 1 - Company Overview...
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-03T02:38:14+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-03T02:38:14+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Company_Handbook.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Company_Handbook.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-03T02:38:14+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-03T02:38:14+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Company_Handbook.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Company_Handbook.pdf', 'file_type': 'pdf'}, page_content='Page 1 - Company Overview'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-03T02:38:14+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-03T02:38:14+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\Company_Handbook.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Company_Handbook.pdf', 'file_type': 'pdf'}, page_content='Company Name: NebulaF

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

e:\Repository\rag-playground\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager(model_name="BAAI/bge-base-en-v1.5")
embedding_manager

Loading embedding model: BAAI/bge-base-en-v1.5


e:\Repository\rag-playground\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BAPS\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4202.07it/s]


Model loaded successfully. Embedding dimension: 768


C:\Users\BAPS\AppData\Local\Temp\ipykernel_13700\1645173577.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [35]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Delete collection if it already exists
            try:
                self.client.delete_collection(self.collection_name)
                print(f"Deleted existing collection: {self.collection_name}")
            except Exception:
                # Collection doesn't exist yet
                pass

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Deleted existing collection: pdf_documents
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [36]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 43 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.40it/s]


Generated embeddings with shape: (43, 768)
Adding 43 documents to vector store...
Successfully added 43 documents to vector store
Total documents in collection: 43


Retriever Pipeline From VectorStore

In [37]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [39]:
rag_retriever.retrieve("CTO?")

Retrieving documents for query: 'CTO?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.39it/s]

Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'doc_003c88ff_4',
  'content': 'CEO: Aarav Mehta\nCTO: Kavya Rao',
  'metadata': {'moddate': '2026-06-03T02:38:14+00:00',
   'page': 0,
   'creationdate': '2026-06-03T02:38:14+00:00',
   'total_pages': 4,
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'title': '(anonymous)',
   'source': '..\\data\\pdf\\Company_Handbook.pdf',
   'file_type': 'pdf',
   'keywords': '',
   'doc_index': 4,
   'creator': '(unspecified)',
   'source_file': 'Company_Handbook.pdf',
   'author': '(anonymous)',
   'content_length': 31,
   'subject': '(unspecified)',
   'page_label': '1',
   'trapped': '/False'},
  'similarity_score': 0.43689143657684326,
  'distance': 0.5631085634231567,
  'rank': 1},
 {'id': 'doc_e3d04fbb_23',
  'content': 'Page 3 - Operations',
  'metadata': {'producer': 'ReportLab PDF Library - www.reportlab.com',
   'source': '..\\data\\pdf\\Company_Handbook.pdf',
   'creator': '(unspecified)',
   'source_file': 'Company_Handbook.pdf',
   'total_pages': 4,
   'page_la

In [40]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="llama3.2:latest",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello!


In [42]:
import os
from openai import OpenAI
from langchain_core.prompts import PromptTemplate

class OllamaLLM:
    def __init__(
        self,
        model_name: str = "llama3.2:latest",
        base_url: str = "http://localhost:11434/v1"
    ):
        """
        Initialize Ollama LLM

        Args:
            model_name: Ollama model name
            base_url: Ollama OpenAI-compatible endpoint
        """

        self.model_name = model_name

        self.client = OpenAI(
            base_url=base_url,
            api_key="ollama"  # Required but can be any string
        )

        print(f"Initialized Ollama LLM with model: {self.model_name}")

    def generate_response(
        self,
        query: str,
        context: str,
        max_length: int = 500
    ) -> str:
        """
        Generate response using retrieved context
        """

        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""
You are a helpful AI assistant.

Use the following context to answer the question accurately and concisely.

Context:
{context}

Question:
{question}

Answer:
If the context does not contain enough information, say so.
"""
        )

        formatted_prompt = prompt_template.format(
            context=context,
            question=query
        )

        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "user",
                        "content": formatted_prompt
                    }
                ],
                max_tokens=max_length,
                temperature=0.1
            )

            return response.choices[0].message.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(
        self,
        query: str,
        context: str
    ) -> str:

        simple_prompt = f"""
Based on this context:

{context}

Question:
{query}

Answer:
"""

        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "user",
                        "content": simple_prompt
                    }
                ],
                temperature=0.1
            )

            return response.choices[0].message.content

        except Exception as e:
            return f"Error: {str(e)}"

In [43]:
llm = OllamaLLM(model_name="llama3.2:latest")

answer = llm.generate_response(
    query="What is RAG?",
    context="Retrieval Augmented Generation combines retrieval with LLM generation."
)

print(answer)

Initialized Ollama LLM with model: llama3.2:latest
RAG stands for Retrieval Augmented Generation. It's a technique that combines retrieval with Large Language Model (LLM) generation to improve the accuracy and efficiency of text generation tasks.


Integration Vectordb Context pipeline With LLM output

In [45]:
def rag_simple(query, retriever, llm, top_k=3):
    """
    Simple RAG pipeline:
    1. Retrieve relevant chunks
    2. Generate answer using OllamaLLM
    """

    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Generate answer using our OllamaLLM class
    response = llm.generate_response(
        query=query,
        context=context
    )

    return response

In [54]:
llm = OllamaLLM(
    model_name="llama3.2:latest"
)

answer = rag_simple(
    query="Who owns FalconStream?",
    retriever=rag_retriever,
    llm=llm
)

print(answer)

Initialized Ollama LLM with model: llama3.2:latest
Retrieving documents for query: 'Who owns FalconStream?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.79it/s]

Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)


I don't have any additional information about who owns FalconStream. The provided context only mentions the internal tracking code and project details, but it doesn't mention the ownership of the platform.
